# Task 3: Multi-Class Image Classification with a CNN
**Progree - Machine Learning Internship**

**Objective:** Train a deep learning vision pipeline on raw pixel arrays.

**Dataset:** CIFAR-10 (60,000 colour images of 32x32 pixels, 10 classes).

**Framework:** TensorFlow / Keras.

**What this notebook does:** custom convolutional, max-pooling and fully connected layers, data augmentation to reduce overfitting, training and validation accuracy curves, and a comparison with and without augmentation.

> **Before running:** Runtime -> Change runtime type -> **T4 GPU**. On CPU training is very slow.

## 1. Imports and GPU check

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import classification_report, confusion_matrix

SEED = 42
keras.utils.set_random_seed(SEED)
print('TensorFlow:', tf.__version__)
print('GPU found :', tf.config.list_physical_devices('GPU'))

## 2. Load and inspect the data

In [ ]:
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()
y_train, y_test = y_train.flatten(), y_test.flatten()
class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

print('Train:', x_train.shape, y_train.shape)
print('Test :', x_test.shape, y_test.shape)
print('Pixel range:', x_train.min(), 'to', x_train.max())

plt.figure(figsize=(12, 5))
for i in range(20):
    plt.subplot(4, 5, i + 1)
    plt.imshow(x_train[i]); plt.title(class_names[y_train[i]], fontsize=9); plt.axis('off')
plt.suptitle('Sample training images'); plt.tight_layout()
plt.savefig('t3_samples.png', dpi=130); plt.show()

## 3. Preprocessing
Pixels are scaled to the range 0-1. Labels stay as integers, so the loss is `sparse_categorical_crossentropy`.

In [ ]:
x_train_n = x_train.astype('float32') / 255.0
x_test_n = x_test.astype('float32') / 255.0
print('Scaled range:', x_train_n.min(), 'to', x_train_n.max())

## 4. Model architecture
Three convolution blocks (Conv -> BatchNorm -> Conv -> BatchNorm -> MaxPool -> Dropout) followed by fully connected layers. Data augmentation is the first part of the model and is active only during training.

In [ ]:
def make_model(augment=True):
    inputs = keras.Input(shape=(32, 32, 3))
    x = inputs
    if augment:
        x = layers.RandomFlip('horizontal')(x)
        x = layers.RandomRotation(0.08)(x)
        x = layers.RandomZoom(0.1)(x)
        x = layers.RandomTranslation(0.1, 0.1)(x)

    for filters, drop in [(32, 0.2), (64, 0.3), (128, 0.4)]:
        x = layers.Conv2D(filters, 3, padding='same', activation='relu')(x)
        x = layers.BatchNormalization()(x)
        x = layers.Conv2D(filters, 3, padding='same', activation='relu')(x)
        x = layers.BatchNormalization()(x)
        x = layers.MaxPooling2D()(x)
        x = layers.Dropout(drop)(x)

    x = layers.Flatten()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(10, activation='softmax')(x)

    model = keras.Model(inputs, outputs)
    model.compile(optimizer=keras.optimizers.Adam(1e-3),
                  loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

model = make_model(augment=True)
model.summary()

## 5. Train the main model (with data augmentation)

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=7, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-5),
]
history = model.fit(x_train_n, y_train, epochs=40, batch_size=64,
                    validation_split=0.1, callbacks=callbacks)

## 6. Training and validation curves

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(history.history['accuracy'], label='training accuracy')
ax[0].plot(history.history['val_accuracy'], label='validation accuracy')
ax[0].set_title('Accuracy'); ax[0].set_xlabel('epoch'); ax[0].legend()
ax[1].plot(history.history['loss'], label='training loss')
ax[1].plot(history.history['val_loss'], label='validation loss')
ax[1].set_title('Loss'); ax[1].set_xlabel('epoch'); ax[1].legend()
plt.tight_layout(); plt.savefig('t3_curves.png', dpi=150); plt.show()

## 7. Test evaluation

In [ ]:
test_loss, test_acc = model.evaluate(x_test_n, y_test, verbose=0)
print(f'Test accuracy: {test_acc:.4f}')

pred = np.argmax(model.predict(x_test_n, verbose=0), axis=1)
print(classification_report(y_test, pred, target_names=class_names))

plt.figure(figsize=(8, 6.5))
sns.heatmap(confusion_matrix(y_test, pred), annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted'); plt.ylabel('True'); plt.title('Confusion matrix (test set)')
plt.tight_layout(); plt.savefig('t3_confusion.png', dpi=150); plt.show()

## 8. Misclassified examples

In [ ]:
wrong = np.where(pred != y_test)[0]
print('Misclassified:', len(wrong), 'of', len(y_test))
plt.figure(figsize=(12, 5))
for k, idx in enumerate(wrong[:15]):
    plt.subplot(3, 5, k + 1)
    plt.imshow(x_test[idx])
    plt.title(f'pred: {class_names[pred[idx]]}\ntrue: {class_names[y_test[idx]]}', color='red', fontsize=8)
    plt.axis('off')
plt.tight_layout(); plt.savefig('t3_errors.png', dpi=130); plt.show()

## 9. Does augmentation reduce overfitting? (comparison run)
The same architecture is trained twice for the same number of epochs, once without and once with augmentation. If augmentation helps, the gap between training and validation accuracy should be smaller with it. Set `RUN_COMPARISON = False` to skip (saves a few minutes).

In [ ]:
RUN_COMPARISON = True
EPOCHS_CMP = 15

if RUN_COMPARISON:
    hists = {}
    for label, aug in [('no augmentation', False), ('with augmentation', True)]:
        m = make_model(augment=aug)
        hists[label] = m.fit(x_train_n, y_train, epochs=EPOCHS_CMP, batch_size=64,
                             validation_split=0.1, verbose=0).history
        gap = hists[label]['accuracy'][-1] - hists[label]['val_accuracy'][-1]
        print(f"{label:18s} final train acc {hists[label]['accuracy'][-1]:.3f} | "
              f"val acc {hists[label]['val_accuracy'][-1]:.3f} | gap {gap:+.3f}")

    plt.figure(figsize=(7, 4.5))
    for label, color in [('no augmentation', '#d62728'), ('with augmentation', '#2a7de1')]:
        plt.plot(hists[label]['accuracy'], '--', color=color, label=label + ' (train)')
        plt.plot(hists[label]['val_accuracy'], '-', color=color, label=label + ' (validation)')
    plt.xlabel('epoch'); plt.ylabel('accuracy'); plt.title('Effect of data augmentation')
    plt.legend(fontsize=8); plt.tight_layout()
    plt.savefig('t3_augmentation_comparison.png', dpi=150); plt.show()

## 10. Save the model

In [ ]:
model.save('cifar10_cnn.keras')
try:
    from google.colab import files
    files.download('cifar10_cnn.keras')
except ImportError:
    print('Saved cifar10_cnn.keras (not running in Colab, so no download prompt).')

## 11. Conclusion
- A multi-layer CNN (6 convolution layers, 3 max-pooling layers, 2 fully connected layers) was trained on raw CIFAR-10 pixels.
- Data augmentation (flip, rotation, zoom, translation), dropout and batch normalisation were used against overfitting, and the comparison in section 9 shows their effect on the train/validation gap.
- Most errors involve visually similar classes (cat/dog, automobile/truck); see the confusion matrix.
- **Future work:** transfer learning (e.g. MobileNetV2), learning-rate schedules, or a deeper ResNet-style network.